# 第8章 カーネルPCAと表現定理 ― デモノートブック

第7章で構成した RKHS を道具として使う章である。鍵は二つ——
内積を $k$ に取り替えるだけで非線形化できること（カーネルトリック）と、
RKHS 上の正則化つき最小化の解が必ず $\sum_i\alpha_ik(\cdot,\boldsymbol{x}_i)$ の形になること（表現定理）。
このノートブックでは、講義ノート第8章の主要な主張——中心化 Gram 行列 $\tilde{\boldsymbol{K}}=\boldsymbol{H}\boldsymbol{K}\boldsymbol{H}$、
カーネルPCAの有限次元化、帯域幅の両極限、表現定理、LOO の閉形式、Nyström と RFF——を
すべて数値で確かめる。**実際に計算するのは $n\times n$ の行列だけ**である。

## 目次

1. [8.1 中心化 Gram 行列](#sec81)（§8.3、命題「中心化 Gram 行列」「テスト点の中心化」）
2. [8.2 カーネルPCA](#sec82)（§8.4、定理「カーネルPCAの有限次元化」）
3. [8.3 帯域幅 $\sigma$ の効き方](#sec83)（§8.4.2）
4. [8.4 表現定理の数値的確認](#sec84)（§8.2、表現定理）
5. [8.5 カーネルリッジ回帰と LOO の閉形式](#sec85)（§8.5、命題「LOO の閉形式」）
6. [8.6 Nyström 近似とランダムフーリエ特徴](#sec86)（§8.9）
7. [8.7 逆像問題](#sec87)（§8.4.3）
8. [演習](#ex) / [演習の解答](#sol)

> 講義ノートの定理・命題には番号を振っているが、章内の連番は版によって前後するため、
> 以下では**節番号と定理名**で引く。

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

### 共通の道具

本書の規約どおり**データ行列は列がサンプル**（$\boldsymbol{X}\in\mathbb{R}^{d\times n}$）である。
scikit-learn は行がサンプルなので、渡すときに `X.T` と転置する。

In [ ]:
from sklearn.datasets import make_circles, make_moons
from scipy.spatial.distance import pdist


def sqdist(A, B):
    """A: d×n, B: d×m -> n×m の二乗距離行列。"""
    return (A * A).sum(0)[:, None] + (B * B).sum(0)[None, :] - 2 * A.T @ B


def rbf(A, B, s):
    """ガウスカーネル。A: d×n, B: d×m -> n×m。"""
    return np.exp(-sqdist(A, B) / (2 * s**2))


def median_sigma(X):
    """メディアンヒューリスティック（pdist は n×d を要求するので転置して渡す）。"""
    return float(np.median(pdist(X.T)))


def center_train(K):
    """K̃ = H K H。"""
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    return H @ K @ H


def center_test(Kn, K):
    """テスト点の中心化。Kn: m×n（テスト×訓練）、K: n×n（訓練 Gram）。
    引くのは訓練データの平均であって、テスト集合の平均ではない。"""
    return (Kn - Kn.mean(1, keepdims=True)
            - K.mean(0, keepdims=True) + K.mean())


print("道具の定義完了")

<a id="sec81"></a>
## 8.1 中心化 Gram 行列

§8.3 の命題「中心化 Gram 行列」は $\tilde{K}_{ij}=\langle\tilde\phi(\boldsymbol{x}_i),\tilde\phi(\boldsymbol{x}_j)\rangle$ が
$\tilde{\boldsymbol{K}}=\boldsymbol{H}\boldsymbol{K}\boldsymbol{H}$（$\boldsymbol{H}=\boldsymbol{I}_n-\frac1n\boldsymbol{1}\boldsymbol{1}^\top$）と書けることを主張し、
続く注意は $\tilde{\boldsymbol{K}}\boldsymbol{1}=\boldsymbol{0}$、$\mathrm{rank}\,\tilde{\boldsymbol{K}}\le n-1$ を指摘する。
まず §8.3 の $3\times3$ の手計算例を再現する。

In [ ]:
K3 = np.array([[1.0, 0.5, 0.0],
               [0.5, 1.0, 0.5],
               [0.0, 0.5, 1.0]])
Kt3 = center_train(K3)
print("9·K̃ =\n", np.round(9 * Kt3, 10))
print("各行の和 =", np.round(Kt3.sum(1), 12), " ← K̃1 = 0")
ev3 = np.linalg.eigvalsh(Kt3)[::-1]
print("固有値 =", np.round(ev3, 6), "  rank =", int((ev3 > 1e-12).sum()), "≤ n-1 = 2")

$9\tilde{\boldsymbol{K}}=\begin{pmatrix}5&-1&-4\\-1&2&-1\\-4&-1&5\end{pmatrix}$、
固有値は $1,\ 1/3,\ 0$ で講義ノート §8.3 の手計算と一致する。
各行の和が $0$（$\tilde{\boldsymbol{K}}\boldsymbol{1}=\boldsymbol{0}$）であることが実装のよい検算になる。

次に §8.3 の命題「テスト点の中心化」を確かめる。警告どおり、
テスト点では $\boldsymbol{K}_\ast$ の行平均を引くだけでは足りず、
**訓練 Gram の列平均と全体平均**も必要である。単体テストは
「訓練点を入れると $\boldsymbol{H}\boldsymbol{K}\boldsymbol{H}$ の対応する行が再現される」ことである。

In [ ]:
rng = np.random.default_rng(0)
Xtr = rng.normal(size=(2, 60))                 # d×n（列がサンプル）
Xte = rng.normal(size=(2, 5))
s0 = median_sigma(Xtr)
K = rbf(Xtr, Xtr, s0)
Kt = center_train(K)

full = center_test(rbf(Xtr, Xtr, s0), K)       # 訓練点をテスト点として入れる
print("σ (median) =", round(s0, 4))
print("単体テスト  max|center_test(K,K) - HKH| =", np.abs(full - Kt).max())

wrong = rbf(Xtr, Xtr, s0) - rbf(Xtr, Xtr, s0).mean(1, keepdims=True)   # 行平均だけ
print("行平均だけ引いた場合の食い違い  max|・- HKH| =", np.abs(wrong - Kt).max())

# テスト集合の大きさで結果が変わってはいけない
kt_a = center_test(rbf(Xte, Xtr, s0), K)
kt_b = center_test(rbf(Xte[:, :2], Xtr, s0), K)
print("テスト集合を 5 点→2 点にしても同じ値か:",
      np.allclose(kt_a[:2], kt_b))

訓練点を入れた結果は $\boldsymbol{H}\boldsymbol{K}\boldsymbol{H}$ と $7.8\times10^{-16}$ の水準で一致する。
一方、行平均だけを引く実装は $0.350$ もずれる——**これが実装で最も多い誤り**である。
最後の行は、中心化に使う量が訓練データだけから決まっているかの確認で、
テスト集合を 5 点から 2 点に減らしても値が変わらないことを見ている。

<a id="sec82"></a>
## 8.2 カーネルPCA

§8.4 の定理「カーネルPCAの有限次元化」により、共分散作用素の固有値問題は
$\tilde{\boldsymbol{K}}\boldsymbol{\alpha}=n\lambda\boldsymbol{\alpha}$ に帰着し、正規化は
$\|\boldsymbol{v}\|_{\mathcal{H}}=1\iff\|\boldsymbol{\alpha}\|=1/\sqrt{\mu}$（$\mu=n\lambda$）である。
講義ノートの警告どおり `eigh` が返す $\|\boldsymbol{\alpha}\|=1$ の固有ベクトルを
**$\sqrt{\mu}$ で割る**のを忘れるとスコアの尺度が狂う。
講義ノートのコードリスト（カーネルPCAの数式どおりの実装）に沿って実装し、
`KernelPCA` と一致することを確かめる。

In [ ]:
from sklearn.decomposition import KernelPCA


def kpca_fit(X, k, s):
    """X は d×n（列がサンプル）。戻り値 A は n×k（列が α_j）。"""
    K = rbf(X, X, s)
    Kt = center_train(K)
    lam, U = np.linalg.eigh(Kt)                # 昇順
    lam, U = lam[::-1][:k], U[:, ::-1][:, :k]
    return K, Kt, lam, U / np.sqrt(lam)        # ‖α‖² = 1/μ


def kpca_transform(Xnew, X, K, A, s):
    """テスト点のスコア（k×m、列がサンプル）。"""
    return (center_test(rbf(Xnew, X, s), K) @ A).T


Xnd, lab_c = make_circles(n_samples=200, factor=0.3, noise=0.05, random_state=0)
Xc = Xnd.T                                     # make_circles は n×d を返すので転置
sig, kk = 0.5, 2
Kc, Ktc, mu, A = kpca_fit(Xc, kk, sig)
Z = A.T @ Ktc                                  # k×n（第 i 列がサンプル i のスコア）

ref = KernelPCA(n_components=kk, kernel="rbf", gamma=1 / (2 * sig**2))
Zref = ref.fit_transform(Xc.T).T               # sklearn は n×d 規約
Xnew = np.array([[0.0, 1.0], [0.0, 0.0]])      # テスト点も d×2
print("scores  :", np.allclose(np.abs(Z), np.abs(Zref)))
print("variance:", np.allclose((Z**2).sum(1), mu), "  (Σ_i z_ij² = μ_j)")
print("test    :", np.allclose(np.abs(kpca_transform(Xnew, Xc, Kc, A, sig)),
                               np.abs(ref.transform(Xnew.T).T)))
print("μ =", np.round(mu, 4), "  寄与率 =", np.round(mu / np.trace(Ktc), 4))

三行とも `True` である。二行目 $\sum_iz_{ij}^2=\mu_j$ が通れば正規化が正しい。
`sklearn` の `gamma` は $\exp(-\gamma\|\boldsymbol{x}-\boldsymbol{y}\|^2)$ の $\gamma=1/(2\sigma^2)$ で、
一致は符号を除いてである（固有ベクトルの符号は不定）。

次に同心円データと三日月データで線形PCAと比較する（§8.4 の図に対応）。

In [ ]:
from sklearn.decomposition import PCA

Xmd, lab_m = make_moons(n_samples=200, noise=0.08, random_state=0)
Xm = Xmd.T
data = [("同心円", Xc, lab_c, 0.5), ("三日月", Xm, lab_m, 0.3)]

fig, axes = plt.subplots(2, 3, figsize=(11.5, 6.4))
for r, (name, Xd, lb, sd) in enumerate(data):
    Kd, Ktd, mud, Ad = kpca_fit(Xd, 2, sd)
    Zd = Ad.T @ Ktd
    Zl = PCA(n_components=2).fit_transform(Xd.T).T     # 線形PCA（k×n に転置）
    for ax, (P, ttl) in zip(axes[r], [
            (Xd, L(f"{name}：入力空間", f"{name}: input")),
            (Zl, L("線形PCA", "linear PCA")),
            (Zd, L(f"カーネルPCA (σ={sd})", f"kernel PCA (σ={sd})"))]):
        for c, col in [(0, C["blue"]), (1, C["red"])]:
            ax.scatter(P[0, lb == c], P[1, lb == c], s=12, c=col)
        ax.set_title(ttl, fontsize=10)
        ax.set_xlabel(L("第1成分", "PC1")); ax.set_ylabel(L("第2成分", "PC2"))
    axes[r][0].set_xlabel("$x_1$"); axes[r][0].set_ylabel("$x_2$")
    acc = lambda v: max((( v > np.median(v)) == (lb == 1)).mean(),
                        ((v > np.median(v)) == (lb == 0)).mean())
    print(f"{name}: 第1成分を中央値で二分したときの正解率  "
          f"線形PCA {acc(Zl[0]):.3f}   カーネルPCA {acc(Zd[0]):.3f}")
fig.tight_layout(); plt.show()

同心円では線形PCAの第1成分による正解率が $0.510$（当てずっぽうと同じ）なのに対し、
カーネルPCA（$\sigma=0.5$）は $1.000$ で完全に分離する。第1主成分が半径に相当する量を
抽出しているためである。三日月でも $0.740\to0.890$（$\sigma=0.3$）に上がる。
線形PCAは座標の直交変換にすぎないので、回転で分離できない構造は取り出せない。

**中心化しないとどうなるか。** $\boldsymbol{K}$ をそのまま固有分解すると何が起こるかを見る。

In [ ]:
lam_raw, U_raw = np.linalg.eigh(Kc)
lam_raw, U_raw = lam_raw[::-1], U_raw[:, ::-1]
u1 = U_raw[:, 0]
one = np.ones(Kc.shape[0]) / np.sqrt(Kc.shape[0])
print("中心化なし: 上位3固有値 =", np.round(lam_raw[:3], 3))
print("            第1固有ベクトルと 1/√n の内積の絶対値 =", round(abs(u1 @ one), 4))
print("中心化あり: 上位3固有値 =", np.round(np.linalg.eigvalsh(Ktc)[::-1][:3], 3))
print("            K̃1 のノルム =", f"{np.linalg.norm(Ktc @ np.ones(200)):.2e}")

Z_raw = (U_raw[:, :2] / np.sqrt(lam_raw[:2])).T @ Kc      # 中心化なしのスコア
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
for ax, (P, ttl) in zip(axes, [(Z, L("中心化あり $\\tilde{K}=HKH$", "centered")),
                               (Z_raw, L("中心化なし $K$", "uncentered"))]):
    for c, col in [(0, C["blue"]), (1, C["red"])]:
        ax.scatter(P[0, lab_c == c], P[1, lab_c == c], s=12, c=col)
    ax.set_title(ttl, fontsize=11)
    ax.set_xlabel(L("第1成分", "PC1")); ax.set_ylabel(L("第2成分", "PC2"))
fig.tight_layout(); plt.show()

中心化しない $\boldsymbol{K}$ の第1固有ベクトルは $\boldsymbol{1}/\sqrt n$ と大きく重なっており（内積の絶対値 $0.8755$）、
第1固有値 $76.25$ は第2固有値 $24.38$ の 3 倍以上ある。
これは「特徴空間での平均方向」を拾っているだけで、分散の構造ではない。
右図で第1成分がほぼ一方向に張り付き、実質的に 1 成分ぶんを無駄にしていることが見て取れる。
中心化すると第1固有値は $30.11$ に下がり（第2固有値 $24.38$、第3固有値 $23.69$ は
中心化前後でほとんど変わらない）、平均方向は固有値 $0$ に落ちる
（$\|\tilde{\boldsymbol{K}}\boldsymbol{1}\|=9.8\times10^{-14}$）。

<a id="sec83"></a>
## 8.3 帯域幅 $\sigma$ の効き方

§8.4.2 は二つの極限を解析している。
$\sigma\to0$ では $\boldsymbol{K}\to\boldsymbol{I}_n$、$\tilde{\boldsymbol{K}}\to\boldsymbol{H}$ となり
固有値が $1$（重複度 $n-1$）に縮退して主成分に順序がつかない。
$\sigma\to\infty$ では $\tilde{\boldsymbol{K}}=\sigma^{-2}\tilde{\boldsymbol{X}}^\top\tilde{\boldsymbol{X}}+O(\sigma^{-4})$、
すなわち線形PCAに退化する。両方を数値で確かめる。

In [ ]:
sig_list = [0.05, 0.5, 5.0]
Xt_c = Xc - Xc.mean(1, keepdims=True)              # 中心化データ（d×n）
G_lin = Xt_c.T @ Xt_c                              # 線形カーネルの中心化 Gram

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, sd in zip(axes, sig_list):
    Kd, Ktd, mud, Ad = kpca_fit(Xc, 2, sd)
    Zd = Ad.T @ Ktd
    for c, col in [(0, C["blue"]), (1, C["red"])]:
        ax.scatter(Zd[0, lab_c == c], Zd[1, lab_c == c], s=12, c=col)
    ax.set_title(L(f"σ={sd}  (γ={1/(2*sd**2):.3g})", f"σ={sd}"), fontsize=11)
    ax.set_xlabel(L("第1成分", "PC1")); ax.set_ylabel(L("第2成分", "PC2"))
    ev = np.linalg.eigvalsh(Ktd)[::-1]
    rel = np.linalg.norm(sd**2 * Ktd - G_lin) / np.linalg.norm(G_lin)
    print(f"σ={sd:5.2f}: 上位2成分の寄与率 = {ev[:2].sum()/ev.sum():.4f}  "
          f"μ_j > μ_1/2 となる j の個数 = {int((ev > ev[0]/2).sum()):3d}  "
          f"‖σ²K̃ - X̃ᵀX̃‖/‖X̃ᵀX̃‖ = {rel:.4f}")
fig.tight_layout(); plt.show()

print("\nメディアンヒューリスティック σ_med =", round(median_sigma(Xc), 4))

$\sigma=0.05$ では上位 2 成分の寄与率がわずか $0.0540$、$\mu_j>\mu_1/2$ を満たす $j$ が $13$ 個もある。
$\tilde{\boldsymbol{K}}\to\boldsymbol{H}$（固有値 $1$ が重複度 $n-1$）への縮退が始まっており、
主成分に順序がつかず埋め込みは無情報な雲になる（左図）。
$\sigma=5$ では
$\|\sigma^2\tilde{\boldsymbol{K}}-\tilde{\boldsymbol{X}}^\top\tilde{\boldsymbol{X}}\|/\|\tilde{\boldsymbol{X}}^\top\tilde{\boldsymbol{X}}\|=0.0394$、
すなわち $\tilde{\boldsymbol{K}}$ は $\sigma^{-2}\tilde{\boldsymbol{X}}^\top\tilde{\boldsymbol{X}}$ に $4\%$ で一致し、
上位 2 成分の寄与率も $0.9833$——埋め込みは線形PCA（8.2 節の中段）とほとんど同じである。
中間の $\sigma=0.5$（寄与率 $0.3954$、$\mu_j>\mu_1/2$ は $3$ 個）でだけ非線形構造が現れる。

なおこのデータのメディアンヒューリスティックは $\sigma_{\mathrm{med}}=0.9085$ で、
分離が起きる $0.5$ 付近より大きい。**「非線形化すれば自動的に分離する」のではなく、
分離が起きる $\sigma$ の窓がある**——三日月データで確かめる。

In [ ]:
print(" σ        正解率（第1成分を中央値で二分）")
lb = lab_m
acc = lambda v: max(((v > np.median(v)) == (lb == 1)).mean(),
                    ((v > np.median(v)) == (lb == 0)).mean())
Zl_m = PCA(n_components=2).fit_transform(Xm.T).T
print(f"線形PCA    {acc(Zl_m[0]):.3f}")
for sd in [0.1, 0.2, 0.3, 0.5, median_sigma(Xm), 3.0]:
    _, Ktd, _, Ad = kpca_fit(Xm, 2, sd)
    tag = "  ← median" if abs(sd - median_sigma(Xm)) < 1e-12 else ""
    print(f"σ={sd:6.3f}   {acc((Ad.T @ Ktd)[0]):.3f}{tag}")

三日月データでは、線形PCA $0.740$、メディアン $\sigma=1.155$ で $0.760$ とほとんど変わらない。
$\sigma$ を median の $1/6$ 程度（$0.2$）まで小さくして $0.990$ に達し、
さらに小さい $\sigma=0.1$ では $0.620$ と今度は点が孤立して悪化する。
$\sigma=3$ では $0.740$ と線形PCAに戻る。
**メディアンヒューリスティックは出発点であって最適値ではない**。
教師なしの場面では下流タスクか、median の数倍・数分の一の範囲での安定性を見て決める。

<a id="sec84"></a>
## 8.4 表現定理の数値的確認

§8.2 の表現定理は、$\min_{f\in\mathcal{H}}L((f(\boldsymbol{x}_i))_i)+\Omega(\|f\|_{\mathcal{H}})$ の
最小解が $f^\star=\sum_{i=1}^n\alpha_ik(\cdot,\boldsymbol{x}_i)$ の形をとることを主張する。
証明は二段——(a) 損失項は $\mathcal{H}_n^\perp$ 成分に依存しない、
(b) 罰則項は $\mathcal{H}_n^\perp$ 成分の分だけ真に増える——であった。両方を数値で確かめる。

**(a)(b) の直接確認。** データ点でない $z$ をとり、
$g=k(\cdot,z)-\sum_i\beta_ik(\cdot,\boldsymbol{x}_i)$（$\boldsymbol{\beta}=\boldsymbol{K}^{-1}\boldsymbol{k}_z$）を作ると
$g\in\mathcal{H}_n^\perp$ である。$g(\boldsymbol{x}_i)=0$（損失不変）と
$J(f^\star+tg)=J(f^\star)+\lambda t^2\|g\|^2$（罰則だけ増える）を見る。

In [ ]:
rng = np.random.default_rng(1)
n_r = 15
Xr = np.sort(rng.uniform(-3, 3, n_r))[None, :]        # 1×15（d=1）
yr = np.sin(1.5 * Xr[0]) + 0.15 * rng.normal(size=n_r)
sr, lamr = 0.35, 1e-2
Kr = rbf(Xr, Xr, sr)
print("cond(K) =", f"{np.linalg.cond(Kr):.1e}", " ← 数値的に安定な設定を選ぶ")
alr = np.linalg.solve(Kr + n_r * lamr * np.eye(n_r), yr)      # KRR の解

z = np.array([[1.234]])                                # データ点でない点
kz = rbf(Xr, z, sr).ravel()
beta = np.linalg.solve(Kr, kz)
g_at_X = kz - Kr @ beta                                # g(x_i)
g_norm2 = float(rbf(z, z, sr)[0, 0] - 2 * beta @ kz + beta @ Kr @ beta)
print("max|g(x_i)| =", f"{np.abs(g_at_X).max():.3e}", "  ← 損失は g に依存しない")
print("‖g‖²_H =", round(g_norm2, 6))

J = lambda pred, nrm2: np.mean((yr - pred)**2) + lamr * nrm2
pred0, nrm0 = Kr @ alr, float(alr @ Kr @ alr)
for t in [0.0, 0.5, 1.0]:
    pred = pred0 + t * g_at_X                          # 予測値は変わらない
    nrm2 = nrm0 + t**2 * g_norm2                       # ノルムだけ増える
    print(f"t={t}: 予測値の変化 {np.abs(pred-pred0).max():.2e}   "
          f"‖f‖² = {nrm2:.5f}   J = {J(pred, nrm2):.6f}")

$g(\boldsymbol{x}_i)$ は $3.1\times10^{-16}$、すなわち直交成分を足しても訓練点での値は変わらない（第一段）。
一方 $\|g\|_{\mathcal{H}}^2=0.2243$ なので $\|f\|^2$ は $2.90325\to2.95932\ (t=0.5)\to3.12751\ (t=1)$ と増え、
$J$ も $0.037995\to0.038556\to0.040238$ と単調に増える（第二段）。
$\Omega(t)=\lambda t^2$ は狭義単調増加なので、最小解では直交成分が $0$ でなければならない。

**(c) 基底を増やしても係数は増えない。** より直接的な確認として、
$\mathrm{span}\{k(\cdot,\boldsymbol{x}_i)\}$ に**データ点でない** 4 点 $z_j$ を加えた
$n+4$ 次元の空間で同じ問題を解く。表現定理が正しければ、追加した $z_j$ の係数は $0$ になるはずである。

In [ ]:
Zextra = np.array([[-2.5, -0.4, 1.234, 2.7]])
Xall = np.hstack([Xr, Zextra])                          # 1×(n+4)
K_XA = rbf(Xr, Xall, sr)                                # n×(n+4)：訓練点での基底の値
K_AA = rbf(Xall, Xall, sr)                              # (n+4)×(n+4)：基底どうしの内積

# J(c) = (1/n)‖y - K_XA c‖² + λ cᵀ K_AA c の停留条件を解く
M = K_XA.T @ K_XA / n_r + lamr * K_AA
c = np.linalg.solve(M, K_XA.T @ yr / n_r)
print("追加した z_j の係数 =", np.round(c[n_r:], 12))
print("|c_extra| の最大 =", f"{np.abs(c[n_r:]).max():.3e}")
print("データ点の係数と KRR 解 α の最大差 =", f"{np.abs(c[:n_r] - alr).max():.3e}")

gg = np.linspace(-3.2, 3.2, 400)[None, :]
f_enl = rbf(gg, Xall, sr) @ c
f_krr = rbf(gg, Xr, sr) @ alr
print("関数としての最大差 =", f"{np.abs(f_enl - f_krr).max():.3e}")

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.scatter(Xr[0], yr, s=18, c=C["gray"], label=L("観測", "data"))
ax.plot(gg[0], f_krr, color=C["blue"], lw=2,
        label=L("$n$ 個の基底で解いた解", "solution with $n$ bases"))
ax.plot(gg[0], f_enl, "--", color=C["red"], lw=2,
        label=L("$n+4$ 個の基底で解いた解", "solution with $n+4$ bases"))
ax.scatter(Zextra[0], np.zeros(4), marker="^", s=60, c=C["green"],
           label=L("追加した中心 $z_j$", "extra centers $z_j$"))
ax.set_xlabel("$x$"); ax.legend(fontsize=9)
ax.set_title(L("表現定理：基底を増やしても解は変わらない",
               "representer theorem: extra bases get zero weight"))
fig.tight_layout(); plt.show()

追加した 4 点の係数は最大 $3.4\times10^{-9}$、データ点の係数は
$\boldsymbol{\alpha}=(\boldsymbol{K}+n\lambda\boldsymbol{I})^{-1}\boldsymbol{y}$ と最大 $2.5\times10^{-9}$ で一致し、
関数としては $2.0\times10^{-13}$ しか違わない（図の実線と破線が重なる）。
残差が厳密な $0$ でないのは $\boldsymbol{K}_{AA}$ の条件数（$10^{8}$ 台）による丸めで、
$\sigma$ を大きくして $\boldsymbol{K}$ を悪条件にすると係数レベルの一致は崩れる
（関数としての一致は保たれる）。
**探索空間を広げても最適解は $n$ 個のカーネル関数の張る空間から出ない**——表現定理の主張そのものである。

<a id="sec85"></a>
## 8.5 カーネルリッジ回帰と LOO の閉形式

§8.5 の定理（カーネルリッジ回帰）は
$\min_f\frac1n\sum_i(y_i-f(\boldsymbol{x}_i))^2+\lambda\|f\|_{\mathcal{H}}^2$ の解が
$\boldsymbol{\alpha}=(\boldsymbol{K}+n\lambda\boldsymbol{I})^{-1}\boldsymbol{y}$ であることを述べ、
命題「LOO の閉形式」は $\boldsymbol{S}=\boldsymbol{K}(\boldsymbol{K}+n\lambda\boldsymbol{I})^{-1}$ に対し
$$\mathrm{CV}(\lambda)=\frac1n\sum_i\Bigl(\frac{y_i-(\boldsymbol{S}\boldsymbol{y})_i}{1-S_{ii}}\Bigr)^2$$
を与える。**この閉形式は $\gamma=n\lambda$ を固定した LOO の恒等式**であって、
$1$ 点抜いた問題で $(n-1)\lambda$ に読み替えると値がずれる。三通りを計算して比べる。

In [ ]:
from sklearn.kernel_ridge import KernelRidge

rng = np.random.default_rng(0)
nk = 80
Xk = np.sort(rng.uniform(-3, 3, nk))[None, :]          # 1×80（d=1, n=80）
f0 = lambda t: np.sin(2 * t) + 0.3 * t
yk = f0(Xk[0]) + 0.2 * rng.normal(size=nk)
sk = median_sigma(Xk)
lk = 1e-3
Kk = rbf(Xk, Xk, sk)

alk = np.linalg.solve(Kk + nk * lk * np.eye(nk), yk)
refk = KernelRidge(alpha=nk * lk, kernel="rbf",
                   gamma=1 / (2 * sk**2)).fit(Xk.T, yk)  # sklearn は n×d
print("σ (median) =", round(sk, 4))
print("α が sklearn と一致:", np.allclose(alk, refk.dual_coef_))

S = Kk @ np.linalg.inv(Kk + nk * lk * np.eye(nk))
loo_closed = np.mean(((yk - S @ yk) / (1 - np.diag(S)))**2)


def loo_brute(reg):
    """n 回学習し直す総当たり。reg は 1 点抜いた問題の正則化の重み。"""
    out = np.empty(nk)
    for i in range(nk):
        Xi, yi = np.delete(Xk, i, 1), np.delete(yk, i)
        a = np.linalg.solve(rbf(Xi, Xi, sk) + reg * np.eye(nk - 1), yi)
        out[i] = (yk[i] - rbf(Xk[:, i:i+1], Xi, sk) @ a)[0]**2
    return out.mean()


b_n, b_n1 = loo_brute(nk * lk), loo_brute((nk - 1) * lk)
print(f"(a) 閉形式（nλ）              CV = {loo_closed:.6f}")
print(f"(b) n 回の再学習（nλ のまま）  CV = {b_n:.6f}   (a) との差 {abs(b_n-loo_closed):.2e}")
print(f"(c) n 回の再学習（(n-1)λ）     CV = {b_n1:.6f}   (a) との差 {abs(b_n1-loo_closed):.2e}")

(a) 閉形式と (b) 総当たり（$n\lambda$ のまま）はともに $0.160705$ で差は $1.3\times10^{-15}$、
(c) $(n-1)\lambda$ に読み替えた総当たりは $0.159708$ で $1.0\times10^{-3}$ ずれる。
証明の「$y_i$ を予測値で置き換えた問題の最小解が $f^{(-i)}$ である」という一段が
正則化項を動かさないことを要求するので、**$\gamma=n\lambda$ を固定するという規約は本質的**である。

閉形式は固有分解 $O(n^3)$ 一回で $(\sigma,\lambda)$ のグリッド探索を可能にする。
実効自由度 $\mathrm{df}(\lambda)=\sum_i\mu_i/(\mu_i+n\lambda)$ もあわせて見る。

In [ ]:
sig_grid = sk * np.logspace(np.log10(0.2), np.log10(5), 16)
lam_grid = np.logspace(-6, 0, 25)
best = (np.inf, None, None)
for sg in sig_grid:
    Kg2 = rbf(Xk, Xk, sg)
    mu_g, U_g = np.linalg.eigh(Kg2)
    Uy = U_g.T @ yk
    for lg in lam_grid:
        filt = mu_g / (mu_g + nk * lg)
        Sy = U_g @ (filt * Uy)
        Sii = (U_g**2 @ filt)
        cv = np.mean(((yk - Sy) / (1 - Sii))**2)
        if cv < best[0]:
            best = (cv, sg, lg)
cv_b, sg_b, lg_b = best
mu_b = np.linalg.eigvalsh(rbf(Xk, Xk, sg_b))
df_b = float((mu_b / (mu_b + nk * lg_b)).sum())
print(f"最良 LOO = {cv_b:.6f}  σ = {sg_b:.4f}  λ = {lg_b:.3e}  df = {df_b:.2f}")
print(f"（ノイズ分散 0.2² = 0.04 が LOO の下限の目安）")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
gg2 = np.linspace(-3.2, 3.2, 400)[None, :]
axes[0].scatter(Xk[0], yk, s=14, c=C["gray"], label=L("観測", "data"))
axes[0].plot(gg2[0], f0(gg2[0]), "k-", lw=1.2, label="$f_0$")
for lg, col, ls in [(1e-6, C["red"], ":"), (lg_b, C["blue"], "-"),
                    (1e-1, C["green"], "--")]:
    a = np.linalg.solve(rbf(Xk, Xk, sg_b) + nk * lg * np.eye(nk), yk)
    axes[0].plot(gg2[0], rbf(gg2, Xk, sg_b) @ a, ls, color=col, lw=1.8,
                 label=f"λ={lg:.1e}")
axes[0].set_xlabel("$x$"); axes[0].legend(fontsize=8)
axes[0].set_title(L("λ を変えたときの適合", "fits for several λ"))

dfs = [float((mu_b / (mu_b + nk * lg)).sum()) for lg in lam_grid]
cvs = []
Kb = rbf(Xk, Xk, sg_b)
for lg in lam_grid:
    Sb = Kb @ np.linalg.inv(Kb + nk * lg * np.eye(nk))
    cvs.append(np.mean(((yk - Sb @ yk) / (1 - np.diag(Sb)))**2))
ax2 = axes[1]
ax2.semilogx(lam_grid, dfs, "o-", ms=3, color=C["purple"],
             label=L("実効自由度 df(λ)", "df(λ)"))
ax2.axvline(lg_b, color=C["gray"], ls=":")
ax2.set_xlabel("$\\lambda$"); ax2.set_ylabel(L("実効自由度", "df"))
ax3 = ax2.twinx(); ax3.grid(False)
ax3.semilogx(lam_grid, cvs, "s-", ms=3, color=C["red"], label="LOO")
ax3.set_ylabel(L("LOO 誤差", "LOO error"))
ax2.set_title(L(f"df と LOO（σ={sg_b:.3f}）", f"df and LOO (σ={sg_b:.3f})"))
lines = ax2.get_lines()[:1] + ax3.get_lines()[:1]
ax2.legend(lines, [l.get_label() for l in lines], fontsize=8, loc="center left")
fig.tight_layout(); plt.show()

グリッド探索の最良は $\mathrm{LOO}=0.039259$、$\sigma=0.6951$、$\lambda=5.623\times10^{-4}$、
このとき実効自由度は $10.88$ である。
LOO 誤差がノイズ分散 $0.2^2=0.04$ とほぼ一致する点で止まっているのが正しい挙動で、
これより小さい値が出たら $\lambda$ の規約か中心化を疑うべきである。
左図で $\lambda=10^{-6}$ は観測点を通ろうとして振動し（過学習）、
$\lambda=10^{-1}$ は平坦化して $f_0$ から離れる（過平滑）。
右図のとおり $\mathrm{df}(\lambda)$ は $\lambda\to0$ で $\mathrm{rank}\,\boldsymbol{K}$、
$\lambda\to\infty$ で $0$ に単調減少し、LOO が最小になるのはその中間である。

<a id="sec86"></a>
## 8.6 Nyström 近似とランダムフーリエ特徴

§8.9 は $O(n^3)$ の壁を破る二つの方法を扱う。
**Nyström** は $m$ 個のランドマークを選んで
$\hat{\boldsymbol{K}}=\boldsymbol{K}_{nm}\boldsymbol{K}_{mm}^{+}\boldsymbol{K}_{mn}$ と近似する（データ依存）。
**RFF** は Bochner の定理に基づき
$\boldsymbol{z}(\boldsymbol{x})=\sqrt{2/D}(\cos(\boldsymbol{\omega}_j^\top\boldsymbol{x}+b_j))_j$ という
陽な有限次元特徴を作る（データ非依存）。相対フロベニウス誤差
$\|\hat{\boldsymbol{K}}-\boldsymbol{K}\|_F/\|\boldsymbol{K}\|_F$ を比べる。

In [ ]:
from sklearn.cluster import KMeans

n_f, reps = 400, 5
Xf = np.random.default_rng(0).normal(size=(2, n_f))     # d×n
sf = median_sigma(Xf)
Kf = rbf(Xf, Xf, sf)
nrmK = np.linalg.norm(Kf)
print("σ (median) =", round(sf, 4), "  tr K =", round(float(np.trace(Kf)), 1))


def rff_err(D, seed):
    rg = np.random.default_rng(seed)
    W = rg.normal(0, 1 / sf, size=(D, 2))               # ω ~ N(0, σ⁻²I)
    b = rg.uniform(0, 2 * np.pi, size=(D, 1))
    Zf = np.sqrt(2.0 / D) * np.cos(W @ Xf + b)          # D×n（列がサンプル）
    return np.linalg.norm(Zf.T @ Zf - Kf) / nrmK        # n×n どうしを比べる


def nys_err(m, seed, mode="uniform"):
    rg = np.random.default_rng(seed)
    if mode == "uniform":
        idx = rg.choice(n_f, m, replace=False)
        Kmm, Knm = Kf[np.ix_(idx, idx)], Kf[:, idx]
    else:                                               # k-means 中心をランドマークに
        km = KMeans(n_clusters=m, n_init=3, random_state=seed).fit(Xf.T)
        Cc = km.cluster_centers_.T                      # d×m
        Kmm, Knm = rbf(Cc, Cc, sf), rbf(Xf, Cc, sf)
    Kh = Knm @ np.linalg.lstsq(Kmm, Knm.T, rcond=1e-14)[0]
    return np.linalg.norm(Kh - Kf) / nrmK


Ds = [100, 400, 1600, 6400]
ms = [10, 20, 40, 80]
e_rff = [np.mean([rff_err(D, s) for s in range(reps)]) for D in Ds]
e_nys = [np.mean([nys_err(m, s) for s in range(reps)]) for m in ms]
e_km = [np.mean([nys_err(m, s, "kmeans") for s in range(reps)]) for m in ms]
tail = np.linalg.eigvalsh(Kf)[::-1]
for D, e in zip(Ds, e_rff):
    print(f"RFF      D={D:5d}   相対誤差 = {e:.4f}")
for m, e, ek in zip(ms, e_nys, e_km):
    print(f"Nyström  m={m:5d}   一様 = {ek*0+e:.2e}   k-means = {ek:.2e}   "
          f"固有値の裾 Σ_(i>m)λ_i/‖K‖_F = {tail[m:].sum()/nrmK:.2e}")
slope = np.polyfit(np.log(Ds), np.log(e_rff), 1)[0]
print(f"\nRFF の両対数の傾き = {slope:.3f}  （理論 -1/2）")

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.loglog(Ds, e_rff, "o-", color=C["red"], label=L("RFF（特徴次元 D）", "RFF (D)"))
ax.loglog(ms, e_nys, "s-", color=C["blue"],
          label=L("Nyström 一様（ランドマーク m）", "Nyström uniform (m)"))
ax.loglog(ms, e_km, "^--", color=C["green"], label=L("Nyström k-means", "Nyström k-means"))
ax.loglog(Ds, e_rff[0] * (np.array(Ds) / Ds[0])**-0.5, ":", color=C["gray"],
          label="$O(D^{-1/2})$")
ax.set_xlabel(L("特徴次元 $D$ / ランドマーク数 $m$", "$D$ or $m$"))
ax.set_ylabel(L("相対フロベニウス誤差", "relative Frobenius error"))
ax.legend(fontsize=8)
ax.set_title(L("Gram 行列の近似誤差", "Gram matrix approximation error"))
fig.tight_layout(); plt.show()

RFF の誤差は $D=100,400,1600,6400$ で $0.1257,\ 0.0539,\ 0.0255,\ 0.0148$、
両対数の傾きは $-0.517$ で理論の $O(D^{-1/2})$ に合う（$D$ を 4 倍して誤差が約半分）。
Nyström（一様ランダム）は $m=10,20,40,80$ で
$2.37\times10^{-2},\ 4.21\times10^{-3},\ 3.87\times10^{-4},\ 3.25\times10^{-5}$ と
はるかに速く減り、**$m=20$ の Nyström（$4.21\times10^{-3}$）が $D=6400$ の RFF（$0.0148$）より正確**である。
$k$-means でランドマークを選ぶとさらに
$8.55\times10^{-3},\ 1.51\times10^{-3},\ 2.94\times10^{-5},\ 6.91\times10^{-8}$ まで下がる。

Nyström の誤差は $\boldsymbol{K}$ の固有値の減衰に従うが、
固有値の裾 $\sum_{i>m}\lambda_i$ がそのまま上界になるわけではない（§8.9 の注意）。
実測でも一様ランダムの誤差は裾（$\|\boldsymbol{K}\|_F$ で割った値で
$1.43\times10^{-2},\ 6.64\times10^{-4},\ 1.28\times10^{-6},\ 9.30\times10^{-12}$）より大きく、
$m$ が大きいほど比が開く。選んだランドマークが張る空間 $\mathcal{V}_I$ は
$\boldsymbol{K}$ の上位固有空間とは一般に一致しないからである。

なお $\boldsymbol{K}_{mm}^{+}$ には注意が要る。$m$ が大きいと $\boldsymbol{K}_{mm}$ の条件数が跳ね上がり、
`np.linalg.pinv` の既定の `rcond` では有効な特異値まで捨てて誤差が「悪化」して見える。
ここでは `lstsq(..., rcond=1e-14)` を使っている（実務では $\boldsymbol{K}_{mm}+\varepsilon\boldsymbol{I}$ のリッジ化が定石）。
使い分けは、データが乗るなら Nyström、ストリーミングや分散環境なら RFF である。

<a id="sec87"></a>
## 8.7 逆像問題

§8.4.3 は、特徴空間で射影した点 $\Psi\in\mathcal{H}$ を入力空間に戻す**逆像**（pre-image）が
一般には存在しないことを指摘し、近似逆像
$\boldsymbol{z}^\star=\arg\min_{\boldsymbol{z}}\|\phi(\boldsymbol{z})-\Psi\|_{\mathcal{H}}^2$ の停留条件として
不動点方程式
$$\boldsymbol{z}=\frac{\sum_i\gamma_ie^{-\|\boldsymbol{z}-\boldsymbol{x}_i\|^2/2\sigma^2}\boldsymbol{x}_i}
{\sum_i\gamma_ie^{-\|\boldsymbol{z}-\boldsymbol{x}_i\|^2/2\sigma^2}}$$
を導いた。ノイズを載せた円周データで、上位 $k$ 主成分への射影＋固定点反復による
ノイズ除去を行い、線形PCAと比べる。

In [ ]:
rng = np.random.default_rng(2)
n_p = 200
th = rng.uniform(0, 2 * np.pi, n_p)
Xtrue = np.vstack([np.cos(th), np.sin(th)])            # 真の多様体（単位円）
Xp = Xtrue + 0.12 * rng.normal(size=(2, n_p))          # d×n、ノイズ入り
sp, kp = 0.5, 6
Kp, Ktp, mup, Ap = kpca_fit(Xp, kp, sp)
Zp = Ap.T @ Ktp                                        # k×n のスコア

# 射影 P φ(x) = Σ_i γ_i φ(x_i)、γ_i = 1/n + Σ_j z_j α_ji（α_j ⊥ 1 を使った）
Gam = np.ones((n_p, n_p)) / n_p + Ap @ Zp              # (i,列=点) の γ
Zden = Xp.copy()
for _ in range(30):                                    # 固定点反復（初期値は元の点）
    Wgt = rbf(Zden, Xp, sp).T * Gam                    # n×n
    den = Wgt.sum(0)
    Zden = (Xp @ Wgt) / np.where(np.abs(den) < 1e-12, 1e-12, den)

pca2 = PCA(n_components=1).fit(Xp.T)
Xlin = pca2.inverse_transform(pca2.transform(Xp.T)).T  # 線形PCA(1成分)で再構成
dist = lambda A: np.abs(np.linalg.norm(A, axis=0) - 1.0).mean()
print(f"単位円からの平均距離   ノイズあり {dist(Xp):.4f}  "
      f"→ カーネルPCA逆像 {dist(Zden):.4f}   線形PCA(1成分) {dist(Xlin):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4.0))
circ = np.vstack([np.cos(np.linspace(0, 2*np.pi, 200)),
                  np.sin(np.linspace(0, 2*np.pi, 200))])
for ax, (P, ttl) in zip(axes, [
        (Xp, L("ノイズあり観測", "noisy data")),
        (Zden, L(f"カーネルPCA 逆像（k={kp}, σ={sp}）", "kPCA pre-image")),
        (Xlin, L("線形PCA（1成分）で再構成", "linear PCA (1 comp.)"))]):
    ax.plot(circ[0], circ[1], "-", color=C["gray"], lw=1, label=L("真の円", "true circle"))
    ax.scatter(P[0], P[1], s=10, c=C["blue"])
    ax.set_aspect("equal"); ax.set_title(ttl, fontsize=10)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
axes[0].legend(fontsize=8)
fig.tight_layout(); plt.show()

単位円からの平均距離は、ノイズあり $0.0979$ → カーネルPCA の逆像 $0.0289$ と
$1/3$ 以下に減る。一方、線形PCA を 1 成分で使うと $0.3426$ と悪化する——
線形の主部分空間は直線なので、円周を直線に潰してしまうためである。
カーネルPCAの主部分空間は入力空間では曲がった多様体に対応する、というのがこの図の主張である。

固定点反復は非凸問題の停留条件なので局所解に落ちうる（初期値には元の点を使っている）。
分母が $0$ に近い点では反復が不安定になるため、実装では下限で保護している。

<a id="ex"></a>
## 演習

**問 1（テスト点の中心化）**
$\boldsymbol{X}\in\mathbb{R}^{2\times50}$ を標準正規乱数（`default_rng(3)`）、$\sigma$ をメディアン規準として、
(a) テスト点の中心化を実装し、訓練点を入れると $\boldsymbol{H}\boldsymbol{K}\boldsymbol{H}$ の対応する行が
再現されることを確かめよ。(b) 訓練 Gram の列平均・全体平均を**テスト集合から計算し直した**
誤った実装を作り、テスト集合の大きさ（5 点と 25 点）で結果が変わることを示せ。

**問 2（LOO の閉形式）**
$n=50$、$x_i\sim\mathrm{Unif}[-2,2]$、$y_i=\cos(3x_i)+0.1\varepsilon_i$ でデータを作り、
$\sigma$ をメディアン規準、$\lambda=10^{-2}$ として
(a) 閉形式 $\mathrm{CV}=\frac1n\sum_i\bigl((y_i-(\boldsymbol{S}\boldsymbol{y})_i)/(1-S_{ii})\bigr)^2$ と
(b) $n$ 回学習し直す総当たり（正則化は $n\lambda$ のまま）が一致すること、
(c) $(n-1)\lambda$ に読み替えるとずれることを確かめよ。

**問 3（ランドマークの選び方）**
8.6 節のデータ（$\boldsymbol{X}\in\mathbb{R}^{2\times400}$、$\sigma$ は median）で $m=20$ とし、
一様ランダム・$k$-means 中心・**列ノルム比例**の 3 通りのランドマーク選択について
Nyström の相対フロベニウス誤差を 5 回の平均で比べよ。どれが良いか、なぜか。

In [ ]:
# 問 1
rng_e = np.random.default_rng(3)
Xe = rng_e.normal(size=(2, 50))
se = median_sigma(Xe)
Ke = rbf(Xe, Xe, se)
Kte = center_train(Ke)
Xe_test = rng_e.normal(size=(2, 25))


def center_test_wrong(Kn, K):
    """誤り：訓練 Gram の列平均・全体平均をテスト集合から計算し直す。"""
    # TODO: Kn - Kn.mean(1,keepdims=True) - Kn.mean(0,keepdims=True) + Kn.mean() を返す
    pass


# TODO: (a) center_test(rbf(Xe,Xe,se), Ke) と Kte の最大差を出力する
# TODO: (b) 誤った実装で Xe_test の先頭 5 点と 25 点全部を渡し、
#           先頭 5 行が一致するかどうかを比べる

In [ ]:
# 問 2
rng_f = np.random.default_rng(4)
nf2 = 50
Xf2 = np.sort(rng_f.uniform(-2, 2, nf2))[None, :]
yf2 = np.cos(3 * Xf2[0]) + 0.1 * rng_f.normal(size=nf2)
sf2, lf2 = median_sigma(Xf2), 1e-3
Kf2 = rbf(Xf2, Xf2, sf2)
# TODO: (a) S = K(K+nλI)⁻¹ から閉形式の CV を計算する
# TODO: (b) i を除いて学習し直す総当たり（正則化は nλ のまま）
# TODO: (c) 正則化を (n-1)λ にしたときの値と比べる

In [ ]:
# 問 3
def nys_err_colnorm(m, seed):
    """列ノルムの二乗に比例した確率でランドマークを選ぶ。"""
    rg = np.random.default_rng(seed)
    p = (Kf**2).sum(0)
    p = p / p.sum()
    idx = rg.choice(n_f, m, replace=False, p=p)
    # TODO: Kmm, Knm を作り、lstsq(rcond=1e-14) で K̂ を組み立てて相対誤差を返す
    return np.nan


# TODO: m=20 で一様・k-means・列ノルム比例の 3 通りを 5 回平均で比べる

<a id="sol"></a>
## 演習の解答

In [ ]:
# --- 問 1 ---
def center_test_wrong(Kn, K):
    return Kn - Kn.mean(1, keepdims=True) - Kn.mean(0, keepdims=True) + Kn.mean()


print("(a) max|center_test(K,K) - HKH| =",
      f"{np.abs(center_test(rbf(Xe, Xe, se), Ke) - Kte).max():.2e}")
Kn25 = rbf(Xe_test, Xe, se)
a25 = center_test_wrong(Kn25, Ke)
a5 = center_test_wrong(Kn25[:5], Ke)
print("(b) 誤った実装：25 点で計算した先頭 5 行と 5 点だけで計算した結果の最大差 =",
      f"{np.abs(a25[:5] - a5).max():.4f}")
c25 = center_test(Kn25, Ke)
c5 = center_test(Kn25[:5], Ke)
print("    正しい実装での同じ差 =", f"{np.abs(c25[:5] - c5).max():.2e}")

(a) 訓練点を入れると $\boldsymbol{H}\boldsymbol{K}\boldsymbol{H}$ と $7.2\times10^{-16}$ の水準で一致する。
(b) 誤った実装ではテスト集合を 25 点から 5 点に減らすだけで結果が $0.1675$ 変わってしまう。
正しい実装では差はちょうど $0$ である。
中心化に使う列平均・全体平均は**訓練データから計算して保存する**量である。

In [ ]:
# --- 問 2 ---
S2 = Kf2 @ np.linalg.inv(Kf2 + nf2 * lf2 * np.eye(nf2))
cv_closed = np.mean(((yf2 - S2 @ yf2) / (1 - np.diag(S2)))**2)


def brute2(reg):
    o = np.empty(nf2)
    for i in range(nf2):
        Xi, yi = np.delete(Xf2, i, 1), np.delete(yf2, i)
        a = np.linalg.solve(rbf(Xi, Xi, sf2) + reg * np.eye(nf2 - 1), yi)
        o[i] = (yf2[i] - rbf(Xf2[:, i:i+1], Xi, sf2) @ a)[0]**2
    return o.mean()


print(f"(a) 閉形式        {cv_closed:.6f}")
print(f"(b) 総当たり nλ    {brute2(nf2*lf2):.6f}   差 {abs(brute2(nf2*lf2)-cv_closed):.2e}")
print(f"(c) 総当たり(n-1)λ {brute2((nf2-1)*lf2):.6f}   差 {abs(brute2((nf2-1)*lf2)-cv_closed):.2e}")

閉形式と総当たり（$n\lambda$）はどちらも $0.079611$ で差は $0$、
$(n-1)\lambda$ に読み替えると $0.078495$ で $1.1\times10^{-3}$ ずれる。
8.5 節と同じ構図で、**閉形式は $\gamma=n\lambda$ を固定した LOO の恒等式**である。

In [ ]:
# --- 問 3 ---
def nys_err_colnorm(m, seed):
    rg = np.random.default_rng(seed)
    p = (Kf**2).sum(0)
    p = p / p.sum()
    idx = rg.choice(n_f, m, replace=False, p=p)
    Kmm, Knm = Kf[np.ix_(idx, idx)], Kf[:, idx]
    Kh = Knm @ np.linalg.lstsq(Kmm, Knm.T, rcond=1e-14)[0]
    return np.linalg.norm(Kh - Kf) / nrmK


m0 = 20
r_uni = np.mean([nys_err(m0, s) for s in range(5)])
r_km = np.mean([nys_err(m0, s, "kmeans") for s in range(5)])
r_cn = np.mean([nys_err_colnorm(m0, s) for s in range(5)])
print(f"m={m0}  一様 {r_uni:.3e}   k-means {r_km:.3e}   列ノルム比例 {r_cn:.3e}")

$m=20$ で一様ランダム $4.21\times10^{-3}$、$k$-means 中心 $1.51\times10^{-3}$、
列ノルム比例 $5.43\times10^{-3}$ である。
$k$-means が最も良いのは、入力空間で散らばった点を選ぶことで
$\mathcal{V}_I=\mathrm{span}\{\phi(\boldsymbol{x}_j):j\in I\}$ の重複を避け、
$\boldsymbol{K}$ の上位固有空間をよく張るからである（§8.9）。
列ノルム比例はこの設定では一様選択よりむしろ悪い。
ガウスカーネルでは列ノルムが密集領域の点で大きくなるので、
似た場所のランドマークばかりを引いて $\mathcal{V}_I$ が重複するためである。
重みの根拠が「上位固有空間をどれだけ担うか」から離れており、
その意味ではレバレッジスコアの方が筋がよい。